# 04 · Árbol, bagging y Random Forest sobre Wine Quality

**Módulo 4 · Sesión 10** — Árboles de decisión y bagging

## Objetivos

El notebook 03 construyó árboles y bagging sobre datos sintéticos en 2D. Este los aplica
a Wine Quality —mismos datos sin duplicados, misma partición y misma validación cruzada
estratificada que `02-clasificacion-aplicado.ipynb`— para responder con números:

1. ¿Un árbol solo, bien podado, compite con la regresión logística? (No.)
2. ¿Cuántos árboles necesita bagging, y sirve el error *out-of-bag* como sustituto de la
   validación cruzada?
3. ¿Random Forest gana a bagging aquí, donde el notebook 03 mostró que no siempre lo hace?
   ¿Qué `max_features` conviene?
4. ¿La diferencia entre el mejor ensamble y la logística sobrevive a la **comparación
   pareada** de `05-sesgo-varianza-validacion.md`, y es relevante?
5. Umbral y conjunto de prueba, con la disciplina del notebook 02.

La métrica principal es la **AP** (área bajo la curva precisión-recall), por las razones de
`03-metricas-clasificacion.md`; se reporta el AUC-ROC al lado.

**Paquetes:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEMILLA = 42

## 1. Mismos datos y misma partición que el notebook 02

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv")
vinos["tipo"] = (vinos["tipo"] == "tinto").astype(int)
vinos = vinos.drop_duplicates().reset_index(drop=True)  # sección 2 del notebook 02
vinos["buena"] = (vinos["quality"] >= 7).astype(int)

X = vinos.drop(columns=["quality", "buena"])
y = vinos["buena"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)


def evaluar_cv(modelo, X=X_train, y=y_train, cv=cv):
    """AP y AUC-ROC promedio de la CV, con error estándar y tiempo de ajuste por pliegue."""
    r = cross_validate(modelo, X, y, cv=cv, scoring=["average_precision", "roc_auc"])
    k = len(r["test_average_precision"])
    return pd.Series(
        {
            "AP": r["test_average_precision"].mean(),
            "ee AP": r["test_average_precision"].std(ddof=1) / np.sqrt(k),
            "AUC-ROC": r["test_roc_auc"].mean(),
            "ajuste (s)": r["fit_time"].mean(),
        }
    )


logistica = Pipeline([("escalar", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])
referencia = evaluar_cv(logistica)
print("Regresión logística (referencia de la sesión 9):")
print(referencia.round(3).to_string())

> El notebook 02 reportó la AP de la logística como 0.517, calculada sobre las
> probabilidades de `cross_val_predict` **juntas**; aquí es 0.526, el **promedio de la AP de
> cada pliegue**. Son dos formas legítimas de agregar; la segunda da además un error
> estándar, que es lo que se necesita para comparar. En este notebook se usa siempre la
> segunda.

Los árboles **no necesitan escalado**: cada partición compara una variable con un umbral, y
el orden no cambia al escalar. Por eso ningún modelo de árboles de este notebook lleva
`StandardScaler`.

## 2. Un árbol solo, podado con criterio

La profundidad es la perilla de complejidad. La misma curva de validación del notebook
03, ahora con AP y validación cruzada en vez de un solo split.

In [ ]:
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
curva_arbol = pd.DataFrame(
    {str(d): evaluar_cv(DecisionTreeClassifier(max_depth=d, random_state=SEMILLA)) for d in profundidades}
).T

plt.figure(figsize=(7, 4))
plt.errorbar(range(len(profundidades)), curva_arbol["AP"], yerr=curva_arbol["ee AP"], fmt="o-", capsize=3, label="árbol")
plt.axhline(referencia["AP"], color="gray", ls="--", label=f"logística ({referencia['AP']:.3f})")
plt.xticks(range(len(profundidades)), [str(d) for d in profundidades])
plt.xlabel("max_depth")
plt.ylabel("AP (validación cruzada)")
plt.title("Un árbol solo nunca alcanza a la regresión logística")
plt.legend()
plt.show()

mejor_prof = curva_arbol["AP"].idxmax()
print(f"Mejor profundidad: {mejor_prof} → AP = {curva_arbol.loc[mejor_prof, 'AP']:.3f} ± {curva_arbol.loc[mejor_prof, 'ee AP']:.3f}")
print(f"Sin límite:        AP = {curva_arbol.loc['None', 'AP']:.3f}")

El mejor árbol solo (profundidad 4) tiene una AP de 0.42: **muy por debajo de la
logística** (0.53). El árbol sin límite, 0.29 — apenas por encima de la prevalencia de 0.19
(el valor de un clasificador aleatorio). Un árbol solo es un modelo débil: sus fronteras en
escalones necesitan muchas particiones para aproximar la relación suave entre `alcohol`,
`density` y la calidad, y cada partición extra memoriza ruido.

## 3. Bagging: cuántos árboles, y el error out-of-bag

Bagging con árboles sin podar. Dos preguntas: cuántos árboles hacen falta, y si la
estimación *out-of-bag* (OOB) —gratis, sin validación cruzada— coincide con la de la CV.

In [ ]:
Bs = [10, 30, 100, 300, 1000]
filas = []
for B in Bs:
    bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=B, oob_score=True, random_state=SEMILLA, n_jobs=-1)
    bag.fit(X_train, y_train)
    p_oob = bag.oob_decision_function_[:, 1]
    con_oob = ~np.isnan(p_oob)  # filas que al menos un árbol dejó fuera
    ap_oob = average_precision_score(y_train[con_oob], p_oob[con_oob])
    r = evaluar_cv(bag)
    filas.append({"B": B, "AP (CV)": r["AP"], "ee": r["ee AP"], "AP (OOB)": ap_oob,
                  "filas sin OOB": int((~con_oob).sum())})
curva_bagging = pd.DataFrame(filas).set_index("B")
print(curva_bagging.round(3).to_string())

Con 10 árboles, bagging ya supera con holgura al mejor árbol solo (0.48 frente a 0.42); con
100 la curva se aplana, y de 300 a 1000 la ganancia (0.002) queda muy dentro del error
estándar — a tres veces el costo. La AP *out-of-bag* sigue a la de validación cruzada a
una o dos centésimas desde $B = 100$, con la ventaja de que se obtiene de un solo ajuste:
para afinar hiperparámetros de un ensamble por bootstrap, OOB es una alternativa legítima a
la CV. Dos precauciones: con pocos árboles, la OOB **subestima** (con $B=10$, cada fila la
predicen en promedio solo $10 \times 0.37 \approx 4$ árboles, y el ensamble evaluado es
mucho más pequeño que el real), y la columna `filas sin OOB` cuenta las filas que ningún
árbol dejó fuera —`oob_decision_function_` les asigna `nan`; con $B=10$ le pasa a cada fila
con probabilidad $0.63^{10} \approx 1\,\%$, con $B \geq 30$ ya no ocurre—.

## 4. Random Forest: ¿decorrelacionar ayuda aquí?

En el notebook 03, con 2 variables informativas de 20, Random Forest **no** superó a
bagging. Wine Quality tiene 12 variables y casi todas llevan algo de señal: es el escenario
donde el muestreo de variables por nodo debería ayudar. Se recorre `max_features` —con 12
variables, `max_features=12` **es** bagging— y `min_samples_leaf`, con $B = 300$.

In [ ]:
rejilla_mf = [1, 2, 3, 4, 6, 12]
rejilla_msl = [1, 2, 5, 10]
resultados_rf = pd.DataFrame(index=rejilla_mf, columns=rejilla_msl, dtype=float)
ee_rf = resultados_rf.copy()
for mf in rejilla_mf:
    for msl in rejilla_msl:
        r = evaluar_cv(RandomForestClassifier(n_estimators=300, max_features=mf, min_samples_leaf=msl, random_state=SEMILLA, n_jobs=-1))
        resultados_rf.loc[mf, msl] = r["AP"]
        ee_rf.loc[mf, msl] = r["ee AP"]

resultados_rf.index.name = "max_features"
resultados_rf.columns.name = "min_samples_leaf"
print("AP de validación cruzada (filas: max_features; columnas: min_samples_leaf):")
print(resultados_rf.round(3).to_string())
print(f"\nError estándar típico: {ee_rf.values.mean():.3f}")

Aquí sí: la fila `max_features=12` (bagging) es la **peor** para cualquier
`min_samples_leaf`, y las de 1–3 variables por nodo, las mejores: cuanta más aleatoriedad,
mejor, hasta el extremo de elegir la variable de cada nodo al azar (`max_features=1`, AP
0.58). La diferencia entre la mejor y la peor fila, ≈0.03 de AP, es el doble del error
estándar (≈0.015); la comparación pareada de la sección 5 lo confirma con más claridad. Y
`min_samples_leaf=1` —árboles completamente desarrollados— es lo mejor en todas las filas
salvo la de bagging, donde empata: en un bosque, los árboles individuales deben ser
complejos; el promedio se encarga de la varianza.

El valor por defecto de `scikit-learn`, `max_features="sqrt"` ($\sqrt{12} \approx 3$), cae
en la zona buena. No es casualidad: es la recomendación original de Breiman para
clasificación, y funciona bien salvo en escenarios como el del notebook 03 — donde el
problema era justamente que casi todas las variables eran ruido.

### Un paso más de aleatoriedad: Extra-Trees

`ExtraTreesClassifier` lleva la idea al extremo: además de muestrear variables, elige el
**umbral** de cada partición al azar en vez de buscar el óptimo. Árboles aún peores
individualmente, aún menos correlacionados, y más rápidos de construir.

In [ ]:
candidatos = {
    "Logística": logistica,
    "Árbol (profundidad 4)": DecisionTreeClassifier(max_depth=4, random_state=SEMILLA),
    "Bagging (B=300)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=300, random_state=SEMILLA, n_jobs=-1),
    "Random Forest (B=300, sqrt)": RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
    "Extra-Trees (B=300, sqrt)": ExtraTreesClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1),
}
tabla = pd.DataFrame({nombre: evaluar_cv(m) for nombre, m in candidatos.items()}).T
print(tabla.round(3).to_string())

## 5. ¿La diferencia sobrevive a la comparación pareada?

Extra-Trees tiene la mejor AP (0.59) y la logística 0.53. Antes de declarar un ganador, la
herramienta de `05-sesgo-varianza-validacion.md`: los dos modelos sobre los **mismos**
pliegues, la diferencia pliegue a pliegue, y su error estándar. Con validación cruzada
repetida (5 pliegues × 4 repeticiones = 20 mediciones) para que el error estándar sea
menos optimista.

In [ ]:
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEMILLA)


def ap_por_pliegue(modelo):
    return cross_validate(modelo, X_train, y_train, cv=cv_rep, scoring="average_precision")["test_score"]


ap_pliegues = {nombre: ap_por_pliegue(m) for nombre, m in candidatos.items() if nombre != "Árbol (profundidad 4)"}


def comparar(a, b):
    d = ap_pliegues[a] - ap_pliegues[b]
    ee = d.std(ddof=1) / np.sqrt(len(d))
    return pd.Series({"diferencia media": d.mean(), "ee": ee, "cociente": d.mean() / ee})


comparaciones = pd.DataFrame(
    {
        "RF − Logística": comparar("Random Forest (B=300, sqrt)", "Logística"),
        "RF − Bagging": comparar("Random Forest (B=300, sqrt)", "Bagging (B=300)"),
        "Extra-Trees − RF": comparar("Extra-Trees (B=300, sqrt)", "Random Forest (B=300, sqrt)"),
    }
).T
print(comparaciones.round(3).to_string())

Las tres diferencias son **detectables** (cociente ≈6, muy por encima de 2): Random Forest
gana a la logística por 0.05 de AP, a bagging por 0.016, y Extra-Trees gana a Random Forest
por 0.018 más. Fíjese en que las dos últimas diferencias tienen un error estándar de 0.003,
**cinco veces menor** que el de cada modelo por separado (≈0.015): es el efecto de
emparejar que `05-sesgo-varianza-intuicion.ipynb` (módulo 3) mostró — los dos modelos
sufren los mismos pliegues difíciles, y al restar eso se cancela.

¿Son **relevantes**? Sobre una AP de 0.53, los +0.06 de Extra-Trees son un 12 % de mejora
relativa — no es el 0.12 % del Ridge-vs-Lasso del módulo 3. Vale la pena; pero no es una
revolución, y la sección 6 lo pone en términos de decisiones. Wine Quality tiene un techo
de ruido alto (la "calidad" es una mediana de tres catadores) que ningún modelo del módulo
va a atravesar.

## 6. Umbral y conjunto de prueba

Igual que en el notebook 02: umbral de mínimo costo ($c_{FN} = 3\,c_{FP}$) elegido con
probabilidades de CV, y después el conjunto de prueba, una sola vez.

In [ ]:
COSTO_FP, COSTO_FN = 1, 3
umbrales = np.round(np.linspace(0.05, 0.95, 91), 2)


def costo_total(y, p, umbral):
    pred = p >= umbral
    return COSTO_FP * np.sum(pred & (y == 0)) + COSTO_FN * np.sum(~pred & (y == 1))


mejor_modelo = ExtraTreesClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1)
p_cv = cross_val_predict(mejor_modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
p_cv_log = cross_val_predict(logistica, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

costos = pd.DataFrame(
    {
        "Extra-Trees": [costo_total(y_train.to_numpy(), p_cv, u) for u in umbrales],
        "Logística": [costo_total(y_train.to_numpy(), p_cv_log, u) for u in umbrales],
    },
    index=umbrales,
)
umbral_optimo = costos["Extra-Trees"].idxmin()

plt.figure(figsize=(7, 4))
plt.plot(costos.index, costos["Extra-Trees"], label="Extra-Trees")
plt.plot(costos.index, costos["Logística"], label="Logística", color="gray")
plt.axvline(umbral_optimo, color="C0", ls=":", label=f"óptimo Extra-Trees ({umbral_optimo:.2f})")
plt.xlabel("Umbral")
plt.ylabel("Costo total (CV)")
plt.legend()
plt.show()

print(f"Costo mínimo en CV — Extra-Trees: {costos['Extra-Trees'].min()} (umbral {umbral_optimo:.2f})   "
      f"Logística: {costos['Logística'].min()} (umbral {costos['Logística'].idxmin():.2f})")

pred_cv = p_cv >= umbral_optimo
print(f"Extra-Trees en el umbral óptimo (CV): precisión {precision_score(y_train, pred_cv):.3f}, "
      f"recall {recall_score(y_train, pred_cv):.3f}, F1 {f1_score(y_train, pred_cv):.3f}")

In [ ]:
mejor_modelo.fit(X_train, y_train)
p_test = mejor_modelo.predict_proba(X_test)[:, 1]
pred_test = (p_test >= umbral_optimo).astype(int)

print(f"Conjunto de prueba, Extra-Trees con umbral {umbral_optimo:.2f}:")
print(f"  AUC-ROC {roc_auc_score(y_test, p_test):.3f}   AP {average_precision_score(y_test, p_test):.3f}")
print(f"  precisión {precision_score(y_test, pred_test):.3f}   recall {recall_score(y_test, pred_test):.3f}   "
      f"F1 {f1_score(y_test, pred_test):.3f}   costo {costo_total(y_test.to_numpy(), p_test, umbral_optimo)}")
print(confusion_matrix(y_test, pred_test))
print(f"\nPara comparar, la logística del notebook 02 sobre el mismo conjunto de prueba: "
      f"AP 0.481, F1 0.521, costo 383.")

En validación cruzada, la ganancia en decisiones es modesta: al mismo recall (≈0.67–0.69),
la precisión pasa de 0.47 a 0.49 y el costo total baja un 4 % (de 1403 a 1341). Sobre el
conjunto de prueba la brecha parece mucho mayor —AP 0.62 frente a 0.48, costo 302 frente a
383—, pero el conjunto de prueba es **una sola muestra** de 1064 vinos: la estimación en la
que hay que confiar para describir la diferencia es la de la CV repetida de la sección 5,
no la del split final, por la misma razón por la que el módulo 3 dejó de fiarse de una
partición única. El test set sirve para confirmar que el número final está en el rango
esperado, no para medir con precisión.

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Un árbol solo compite con la logística? | No: AP 0.42 (profundidad 4) frente a 0.53; sin podar, 0.29 |
| ¿Cuántos árboles? | Con 100 la AP se aplana; 300 → 1000 queda dentro del error estándar |
| ¿OOB sustituye a la CV? | Desde $B \approx 100$, sí (una o dos centésimas de diferencia), con un solo ajuste; con pocos árboles subestima |
| ¿Random Forest gana a bagging aquí? | Sí, 0.016 ± 0.003 de AP: con 12 variables informativas, decorrelacionar ayuda; `max_features=12` es la peor fila y `max_features=1` la mejor |
| ¿Extra-Trees? | Mejor todavía (0.59, +0.018 ± 0.003 sobre RF), y más rápido |
| ¿Detectable y relevante? | RF − logística = +0.05 de AP, cociente 6; un 12 % relativo con Extra-Trees: vale la pena, sin ser una revolución |
| Conjunto de prueba | Extra-Trees mejora a la logística en AP, F1 y costo; la magnitud fiable es la de la CV repetida, no la del split único |

Todos los ensambles de este notebook promedian árboles **independientes** entre sí. La
sesión 11 introduce la otra familia —**boosting**, donde cada árbol corrige los errores del
anterior— y la compara sobre estos mismos datos y pliegues.